# Dynamics-aware FeatureDock — Full PDBBind v2020 Training (Colab GPU)

**Augmentation ①: Dyna-1 µs–ms dynamics channel (6×80 → 6×81)**

This notebook runs the *entire* dynamics-aware FeatureDock pipeline on a Colab GPU:

1. Clone the repos + install dependencies
2. Unpack the Linux **FEATURE** binary + **DSSP** (these run on Linux only — the reason we use Colab, not a Mac)
3. Download **Dyna-1 ESM-2** weights
4. Point at **PDBBind v2020 refined set** (you provide the path — it needs free registration)
5. **For every structure:** run Dyna-1 → featurize to 6×81 → generate occupancy labels
6. Build the **CDK2-as-validation split** (all PDBBind trains, CDK2 held out)
7. Train **baseline (80)** and **+Dyna-1 (81)**, compare on the CDK2 validation set
8. Save checkpoints + logs to Google Drive to import back into the repo

> **Runtime → Change runtime type → GPU (T4 is fine).**

**Scope:** CDK2 is the *validation* set only — excluded from training. Training uses all other PDBBind refined structures (~4,500).

---
## ⚠️ If Colab disconnects mid-run (READ THIS)

Colab wipes **`/content/`** on every reset — repos, installed packages, the extracted
PDBBind, and the unpacked FEATURE binaries all vanish. **Your Drive is untouched**
(all `pvar_80/`, `pvar_81/`, labels, and the PDBBind `.gz` survive).

**To resume:** just re-run the setup cells top-to-bottom —
§1 (mount) → §2 (clone) → §3 (deps) → §4 (place code) → §5 (FEATURE+chmod) →
§6 (weights) → §7 (path) → re-extract PDBBind (copy `.gz` to local, untar) → §8.

The §8 loop is **resumable**: structures already written to Drive show `skip(done)`
and are not recomputed, so it continues where it left off. Takes ~5 min to get back
to processing.

> To make this painless, keep a **patched `create_voxels_and_landmarks.py`** in your
> Drive `code/` folder (§4 copies it in automatically). All other fixes are already
> baked into the cells below.
---

## 0 · Check GPU

In [ ]:
!nvidia-smi -L || echo "NO GPU — set Runtime > Change runtime type > GPU"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1 · Mount Google Drive (data in / results out)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/dynamics_aware_featuredock'
os.makedirs(f'{WORK}/results', exist_ok=True)
print('Working dir on Drive:', WORK)

## 2 · Clone the repos (same layout as the local Hackathon folder)

In [ ]:
%cd /content
!git clone https://github.com/xuhuihuang/featuredock.git 2>/dev/null || (cd featuredock && git pull)
!git clone https://github.com/WaymentSteeleLab/Dyna-1.git 2>/dev/null || (cd Dyna-1 && git pull)
import os; print("repos:", [d for d in os.listdir('/content') if os.path.isdir(d)])

## 3 · Install dependencies
Colab already ships torch+CUDA; we add the FeatureDock / Dyna-1 stack.

In [ ]:
# Complete dependency set (discovered during first run). Colab ships torch+CUDA;
# we protect it by NOT reinstalling torch. numpy stays 1.26.4 for Dyna-1/biotite --
# the red "pip dependency resolver" warning about numpy>=2 is COSMETIC, ignore it.
!pip -q install rdkit prolif MDAnalysis mdtraj biopython scipy scikit-learn pandas gemmi tqdm
!pip -q install pymol-open-source torcheval msgpack msgpack-numpy cloudpickle tenacity
!pip -q install "transformers<4.47.0" "huggingface_hub[cli]" fair-esm "biotite==0.41.2"
# Dyna-1's own requirements, minus any torch line (keep Colab's GPU torch intact)
!cd /content/Dyna-1 && grep -v -E "^torch" requirements.txt > /tmp/dyna1_reqs.txt 2>/dev/null; pip -q install -r /tmp/dyna1_reqs.txt 2>/dev/null

# --- health check (writes a temp file to avoid shell quote-nesting issues) ---
import subprocess
def _check(name, code, cwd=None):
    open('/tmp/_check.py','w').write(code)
    r = subprocess.run('python /tmp/_check.py', shell=True, capture_output=True, text=True, cwd=cwd)
    print(f"{name}:", r.stdout.strip() or r.stderr.strip()[-160:])

_check("pymol", 'import pymol; print(1)')
_check("rdkit", 'from rdkit import Chem; print(1)')
_check("numpy", 'import numpy; print(numpy.__version__)')
_check("dyna1", 'from model.model import *; print("OK")', cwd='/content/Dyna-1')
print("\nExpected: pymol:1  rdkit:1  numpy:1.26.4  dyna1:OK  (red pip warning above is harmless)")

## 4 · Place the augmentation-① code into FeatureDock
The 4 new files from the local build. Upload them to `{WORK}/code/` on Drive (or `git pull` your fork), then run this cell.

In [ ]:
import shutil, os
SRC = f'{WORK}/code'   # put the 4 new .py files here on Drive
FD  = '/content/featuredock'
mapping = {
  'featurize_dyna1_channel.py': f'{FD}/src/curate_dataset/featurize_dyna1_channel.py',
  'make_cdk2_split.py':         f'{FD}/src/curate_dataset/make_cdk2_split.py',
  'train_dynamics_aware.py':    f'{FD}/src/models/train_dynamics_aware.py',
  'parse_config.py':            f'{FD}/src/models/parse_config.py',
}
for name, dst in mapping.items():
    s = os.path.join(SRC, name)
    if os.path.exists(s): shutil.copy(s, dst); print("placed", name)
    else: print("MISSING on Drive:", s, "-> upload it to", SRC)

# NOTE: create_voxels_and_landmarks.py needs the relaxed protein+ligand loaders.
# If you keep a PATCHED copy in your Drive code/ folder, place it too:
_patched = os.path.join(SRC, 'create_voxels_and_landmarks.py')
if os.path.exists(_patched):
    shutil.copy(_patched, f'{FD}/src/curate_dataset/create_voxels_and_landmarks.py')
    print("placed patched create_voxels_and_landmarks.py")
else:
    print("(optional) no patched create_voxels_and_landmarks.py in Drive code/ -- "
          "the repo default works but drops ~15-40% of structures on ligand/protein valence errors")

## 5 · Unpack the FEATURE binary + DSSP (Linux)
FeatureDock ships FEATURE 3.1.0 as a zip and a `dssp` binary. These run on Linux (Colab) but not macOS — the core reason for using Colab.

In [ ]:
# Unpack FEATURE program + make ALL binaries executable (zip loses +x on extract)
%cd /content/featuredock/src/utils
!unzip -o -q feature-3.1.0.zip -d /content/featuredock/src/
FP='/content/featuredock/src/feature-3.1.0'
!chmod +x {FP}/src/featurize {FP}/bin/featurize {FP}/bin/buildmodel {FP}/bin/scoreit 2>/dev/null
!chmod +x /content/featuredock/src/utils/dssp 2>/dev/null
# verify featurize launches (should print usage, NOT "Permission denied")
!cd {FP}/src && ./featurize 2>&1 | head -3
%cd /content

## 6 · Download Dyna-1 ESM-2 weights
From HuggingFace `gelnesr/Dyna-1` (per the Dyna-1 README). The ESM-2 backbone (`facebook/esm2_t6_8M_UR50D`) auto-downloads on first inference — no gated ESM-3 access needed.

In [ ]:
%cd /content/Dyna-1
!mkdir -p model/weights
!huggingface-cli download gelnesr/Dyna-1 --local-dir model/weights/ 2>&1 | tail -3
!ls -la model/weights/
%cd /content

## 7 · Point at PDBBind v2020 (refined set)
PDBBind needs free registration, so it can't auto-download. Download the v2020 *refined set* from http://www.pdbbind.org.cn/ once, put it on Drive, and set `PDBBIND_DIR`.

In [ ]:
# Point PDBBIND_DIR at wherever you extracted PDBBind. Works with either layout:
#   * flat refined set:      PDBBIND_DIR/<pid>/<pid>_protein.pdb
#   * nested general set:    PDBBIND_DIR/<year-range>/<pid>/<pid>_protein.pdb  (the "P-L" download)
# Section 8 auto-detects the nesting and filters to the refined-set ids, so either is fine.
#
# Extract to LOCAL disk (fast), not Drive:
#   !cp "{WORK}/PDBbind_v2020.gz" /content/pdbbind.gz && mkdir -p /content/pdbbind && tar xzf /content/pdbbind.gz -C /content/pdbbind
PDBBIND_DIR = '/content/pdbbind/P-L'   # <-- set to your extracted root

import os
if os.path.isdir(PDBBIND_DIR):
    top = [d for d in os.listdir(PDBBIND_DIR) if os.path.isdir(os.path.join(PDBBIND_DIR, d))]
    # peek one level to report whether it's flat or nested
    sample = top[0] if top else None
    nested = sample and not os.path.exists(os.path.join(PDBBIND_DIR, sample, f'{sample}_protein.pdb'))
    print(f"Root OK: {len(top)} top-level entries. Layout:",
          "NESTED (year-range/pid)" if nested else "FLAT (pid)")
    print("examples:", top[:5])
else:
    print("NOT FOUND:", PDBBIND_DIR, "\n-> extract PDBBind and set PDBBIND_DIR to its root.")

## 8 · Preprocess → voxels → FEATURE → 6×81, per structure (runnable loop)
For **every** complex in the PDBBind refined set this runs FeatureDock's canonical pipeline plus the Dyna-1 step:
1. `prepare_structure.py` → cleaned apo protein (+DSSP) & ligand
2. `create_voxels_and_landmarks.py` → pocket grid **voxels**
3. `featurize.py` → native **(N,480)** into `pvar_80/`
4. `dyna1-esm2.py` → per-residue **p_exchange**
5. `featurize_dyna1_channel.py` → **(N,486)** = 6×81 into `pvar_81/`
6. `label_voxels.py` → occupancy **labels** (written for both widths)

Assumes the standard refined-set layout: `<pid>/<pid>_protein.pdb` + `<pid>_ligand.sdf`. The loop is **resumable** — it skips complexes already featurized — and writes `preprocess_log.json` so any failures are inspectable. This is the bulk of the runtime; everything lands on Drive.

In [ ]:
import os, glob, subprocess, traceback
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.PDBData import protein_letters_3to1_extended as T2O

FD, DY = '/content/featuredock', '/content/Dyna-1'
SCRIPTS = f'{FD}/src/curate_dataset'
FEATURE_PROGRAM = f'{FD}/src/feature-3.1.0'      # unpacked in step 5
DSSP = f'{FD}/src/utils/dssp'

# work dirs (mirror FeatureDock's step2 layout), kept on Drive so the run resumes
APO  = f'{WORK}/apo';   HET = f'{WORK}/het'
VOX  = f'{WORK}/voxels'; FFD = f'{WORK}/ff';  DET = f'{WORK}/voxel_details'
DY_CSV = f'{WORK}/dyna1_csv'
OUT80, OUT81 = f'{WORK}/pvar_80', f'{WORK}/pvar_81'
for d in (APO,HET,VOX,FFD,DET,DY_CSV,OUT80,OUT81):
    os.makedirs(d, exist_ok=True)

def sh(cmd, cwd=None, timeout=1200):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    return r.returncode, r.stdout, r.stderr

# uppercase-keyed 3->1 map (PDB resnames are UPPERCASE); non-canonical -> 'G'
# (Dyna-1 rejects 'X'; 'G' preserves sequence length so per-residue alignment holds)
T2O_UP = {k.upper(): v for k, v in T2O.items()}
VALID = set("ACDEFGHIKLMNPQRSTVWY")
def extract_seq(pdb_path, chain=None):
    model = next(PDBParser(QUIET=True).get_structure('s', pdb_path).get_models())
    seq = []
    for ch in model:
        if chain and ch.id != chain: continue
        for res in ch:
            if is_aa(res, standard=True) and not res.id[0].strip():
                seq.append(T2O_UP.get(res.resname.strip().upper(), 'G'))
    return ''.join(seq)

def process_one(pid, prot_pdb, lig_sdf):
    """Full canonical FeatureDock pipeline + Dyna-1 channel for one complex."""
    # 1) prepare structure: cleaned apo protein (+ DSSP) and heterogen ligand
    rc,o,e = sh(f'python {SCRIPTS}/prepare_structure.py --pdbid={pid} '
                f'--protfile={prot_pdb} --ligfile={lig_sdf} '
                f'--apodir={APO} --hetdir={HET} --dssp={DSSP} --rm_all_het')
    apo = f'{APO}/{pid}.pdb'; het = f'{HET}/{pid}_ligand.sdf'
    if not os.path.exists(apo): return f'prepare_failed: {e[-160:]}'

    # 2) voxels + landmarks (pocket grid); create_voxels writes to DET, then we move
    rc,o,e = sh(f'python {SCRIPTS}/create_voxels_and_landmarks.py --pdbid={pid} '
                f'--apofile={apo} --hetfile={het} --outdir={DET} '
                f'--pocket_cutoff=6.0 --spacing=1.0 --trim --trim_min=1.0 --trim_max=6.0 '
                f'--abs_include --heavyatom --intermediate --overwrite')
    for suf in ('voxels.pkl','landmarks.pkl'):
        src = f'{DET}/{pid}.{suf}'
        if os.path.exists(src): sh(f'mv {src} {VOX}/')
    vox = f'{VOX}/{pid}.voxels.pkl'
    if not os.path.exists(vox): return f'voxels_failed: {e[-160:]}'

    # 3) native FEATURE -> (N,480) into OUT80
    rc,o,e = sh(f'python {SCRIPTS}/featurize.py --pdbid={pid} '
                f'--voxelfile={vox} --voxeldir={OUT80} --tempdir={FFD} '
                f'--searchdir={APO} --featurize={FEATURE_PROGRAM} '
                f'--numshell=6 --width=1.25 --overwrite')
    pvar80 = f'{OUT80}/{pid}.property.pvar'
    if not os.path.exists(pvar80): return f'featurize_failed: {e[-160:]}'

    # 4) Dyna-1 per-residue p_exchange
    seq = extract_seq(apo)
    if not seq: return 'empty_sequence'
    rc,o,e = sh(f'python dyna1-esm2.py --sequence {seq} --name {pid} --save_dir {DY_CSV}', cwd=DY)
    csv = f'{DY_CSV}/{pid}.csv'
    if not os.path.exists(csv): return f'dyna1_failed: {e[-160:]}'

    # 5) append motion channel -> (N,486) into OUT81
    rc,o,e = sh(f'python {SCRIPTS}/featurize_dyna1_channel.py '
                f'--pdb {apo} --voxelfile {vox} --pvarfile {pvar80} '
                f'--dyna1csv {csv} --out {OUT81}/{pid}.property.pvar')
    if not os.path.exists(f'{OUT81}/{pid}.property.pvar'): return f'channel_failed: {e[-160:]}'

    # 6) occupancy labels from cocrystal ligand -> written for BOTH widths
    lm = f'{VOX}/{pid}.landmarks.pkl'
    rc,o,e = sh(f'python {SCRIPTS}/label_voxels.py --pdbid={pid} '
                f'--voxelfile={vox} --lmfile={lm} '
                f'--configfile={SCRIPTS}/label_config.json --hard '
                f'--outdir={OUT80} --interactions HeavyAtomsite --overwrite')
    lab = f'{OUT80}/{pid}.HeavyAtomsite.labels.pkl'
    if not os.path.exists(lab): return f'label_failed: {e[-160:]}'
    sh(f'cp {lab} {OUT81}/')       # same labels apply to the 81-wide tensors
    return 'ok'

# ---- discover complexes: handles BOTH the flat refined-set layout
#      (<pid>/<pid>_protein.pdb) AND the nested general-set layout
#      (P-L/<year-range>/<pid>/...). Filters to the REFINED-SET ids so the
#      CDK2 split + clan graph (built on the refined set) stay valid.
FD = '/content/featuredock'
REFINED_LIST = f'{FD}/data/pdblist.txt'   # 5,316 refined-set ids shipped with the repo
assert os.path.exists(REFINED_LIST), "run the clone cell (section 2) first -- data/pdblist.txt missing"
refined = {l.strip().lower() for l in open(REFINED_LIST) if l.strip()}
print("refined-set ids in list:", len(refined))

def find_complex_dirs(root):
    """Yield (pid, folder) for every complex dir, at depth 1 (flat) or 2 (P-L/year/)."""
    for a in sorted(os.listdir(root)):
        ad = os.path.join(root, a)
        if not os.path.isdir(ad):
            continue
        # depth-1: is this itself a complex folder?
        if os.path.exists(os.path.join(ad, f'{a}_protein.pdb')):
            yield a, ad
        else:
            # depth-2: a year-range (or other) wrapper holding complex folders
            for b in sorted(os.listdir(ad)):
                bd = os.path.join(ad, b)
                if os.path.isdir(bd) and os.path.exists(os.path.join(bd, f'{b}_protein.pdb')):
                    yield b, bd

complexes, skipped_noligand = [], 0
for pid, cdir in find_complex_dirs(PDBBIND_DIR):
    if pid.lower() not in refined:        # keep only refined-set complexes
        continue
    prot = os.path.join(cdir, f'{pid}_protein.pdb')
    lig  = os.path.join(cdir, f'{pid}_ligand.sdf')
    if os.path.exists(prot) and os.path.exists(lig):
        complexes.append((pid, prot, lig))
    else:
        skipped_noligand += 1

print(f'{len(complexes)} refined-set complexes ready '
      f'(filtered from the full download; {skipped_noligand} refined ids missing a .sdf ligand)')
print('examples:', [c[0] for c in complexes[:5]])

# ---- run the loop (resumable: skips complexes already featurized) ----
from tqdm.auto import tqdm
log = {}
for pid, prot, lig in tqdm(complexes):
    if os.path.exists(f'{OUT81}/{pid}.property.pvar') and os.path.exists(f'{OUT81}/{pid}.HeavyAtomsite.labels.pkl'):
        log[pid] = 'skip(done)'; continue
    try:
        log[pid] = process_one(pid, prot, lig)
    except Exception as ex:
        log[pid] = f'exception: {ex}'

ok = sum(1 for v in log.values() if v in ('ok','skip(done)'))
print(f'\nDONE: {ok}/{len(complexes)} succeeded')
from collections import Counter
print('outcomes:', Counter(v.split(":")[0] for v in log.values()))
import json; json.dump(log, open(f'{WORK}/preprocess_log.json','w'), indent=2)
print('log ->', f'{WORK}/preprocess_log.json')

## 9 · Build the CDK2-as-validation split
All featurized structures train; CDK2 held out (homolog-safe scope — no CDK2 relative leaks into train).

In [ ]:
%cd /content/featuredock
!python src/curate_dataset/make_cdk2_split.py \
    --datafolder {OUT81} --datadir /content/featuredock/data \
    --cdk2-scope homologs \
    --out-train {WORK}/train_pids.txt --out-val {WORK}/val_pids.txt
!echo "--- CDK2 validation set ---"; cat {WORK}/val_pids.txt

## 10 · Train BOTH arms (baseline 80 vs +Dyna-1 81), identical everything else

In [ ]:
# Baseline (80 channels)
!python src/models/train_dynamics_aware.py \
    --modeltype transformer --feature_per_shell 80 --datafolder {OUT80} \
    --train-pids {WORK}/train_pids.txt --val-pids {WORK}/val_pids.txt \
    --outfolder {WORK}/results/baseline80 --modelname baseline80 \
    --steps 300 --n_structs 8 --lr 1e-3 --save_every 25 --use_gpu

In [ ]:
# +Dyna-1 (81 channels)
!python src/models/train_dynamics_aware.py \
    --modeltype transformer --feature_per_shell 81 --datafolder {OUT81} \
    --train-pids {WORK}/train_pids.txt --val-pids {WORK}/val_pids.txt \
    --outfolder {WORK}/results/dyna1_81 --modelname dyna1_81 \
    --steps 300 --n_structs 8 --lr 1e-3 --save_every 25 --use_gpu

## 11 · Compare learning curves on the CDK2 validation set

In [ ]:
import json, matplotlib.pyplot as plt
h0 = json.load(open(f'{WORK}/results/baseline80/baseline80_history.json'))
h1 = json.load(open(f'{WORK}/results/dyna1_81/dyna1_81_history.json'))
fig, ax = plt.subplots(1, 2, figsize=(11,4))
for h, lbl, c in [(h0,'baseline (80)','#888888'), (h1,'+Dyna-1 (81)','#2b6cb0')]:
    s=[r['step'] for r in h]
    ax[0].plot(s,[r['val_loss'] for r in h], label=lbl, color=c)
    ax[1].plot(s,[r['val_mcc']  for r in h], label=lbl, color=c)
ax[0].set_title('CDK2 validation loss'); ax[0].set_xlabel('step'); ax[0].set_ylabel('loss'); ax[0].legend()
ax[1].set_title('CDK2 validation MCC');  ax[1].set_xlabel('step'); ax[1].set_ylabel('MCC');  ax[1].legend()
fig.tight_layout(); fig.savefig(f'{WORK}/results/compare_cdk2.png', dpi=150); plt.show()
print("Saved:", f'{WORK}/results/compare_cdk2.png')

## 12 · Import results back into the repo
Under `{WORK}/results/` on Drive you now have:
- `baseline80/baseline80_final_params.torch` + `_history.json`
- `dyna1_81/dyna1_81_final_params.torch` + `_history.json`
- `compare_cdk2.png`

Download that `results/` folder into your local repo at `featuredock/results/`. See **README_dynamics_aware.md → "Importing Colab-trained results"** for loading the checkpoint and running prediction/evaluation on CDK2.